# 📋 規格書（請直接在這一格修改）

學號：411316442 姓名：張芳瑋 卷別：A

## 1. 系統目的（一句話）
讓同學用一句話完成系學會活動的報名、取消與剩餘名額查詢，並支援多人攜伴、個人限報 2 個活動與幹部統計查詢。

## 2. 使用者會怎麼說（至少 5 句）
1. 我要報名迎新晚會，我叫王小明
2. 我和兩個朋友一起報名程式工作坊，我是陳大同
3. 我要取消迎新晚會，名字是王小明
4. 查詢籃球友誼賽剩餘名額
5. 各活動報名狀況
6. 我明天要考試，下午適合去K書嗎？
7. 我要報名演唱會
8. 我叫王小明

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| activities | id, name, date, quota | INTEGER, TEXT, TEXT, INTEGER | 儲存活動名稱、日期與總名額 |
| registrations | id, activity_name, student_name, count, status | INTEGER, TEXT, TEXT, INTEGER, TEXT | 儲存報名紀錄、學生姓名、人數與狀態 |

初始資料：
- activities: 迎新晚會 (2026-11-20, 6人)、程式工作坊 (2026-11-25, 4人)、籃球友誼賽 (2026-11-28, 3人)
- registrations: 空白

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,活動,人數`
- 動作只能是：`報名`、`取消`、`查詢`、`統計`、`未知`
- 活動只能是：`迎新晚會`、`程式工作坊`、`籃球友誼賽`、`未知`
- 人數的規則：請從句子中提取人數（例如「兩個朋友」加自己共 3 人），若未提及則預設為 1 人。人數限制在 1～3 之間。
- 無法判斷時輸出：`未知,未知,1`
- AI 不可以做的事：提供與活動無關的建議、推薦非清單內的活動。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 報名 | 姓名有效、活動存在、未額滿、未重複報名同活動、個人同時報名未達 2 個上限、人數 1~3 | 提示報名成功與剩餘名額 | 新增一筆記錄至 `registrations` (status='已報名') |
| 取消 | 姓名與活動相符且有有效報名紀錄 | 提示取消成功並釋出名額 | 將該筆紀錄的 status 改為 `已取消` |
| 查詢 | 活動存在 | 顯示活動日期、總名額與剩餘名額 | 無變化 |
| 統計 | 使用者要求查看各活動狀況 | 用 JOIN 與 GROUP BY 顯示各活動日期、總名額、已報名人數與剩餘名額（含無人報名活動） | 無變化 |

## 6. 例外情況（至少 3 個）
1. 未提供姓名直接報名或取消，系統拒絕執行並提示。
2. 報名時超過個人最多報名 2 個活動的上限，系統拒絕。
3. 攜伴人數大於 3 人或小於 1 人，程式自動阻擋並提示錯誤。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**

| # | 類型 | 使用者說的話 | 預期動作 | 預期活動 | 預期人數 |
|---|---|---|---|---|---|
| 1 | 正常 | 我要報名迎新晚會，我叫王小明 | 報名 | 迎新晚會 | 1 |
| 2 | 正常 | 我和兩個朋友一起報名程式工作坊，我是陳大同 | 報名 | 程式工作坊 | 3 |
| 3 | 正常 | 我要取消迎新晚會，名字是王小明 | 取消 | 迎新晚會 | 1 |
| 4 | 正常 | 查詢籃球友誼賽剩餘名額 | 查詢 | 籃球友誼賽 | 1 |
| 5 | 正常 | 各活動報名狀況 | 統計 | 未知 | 1 |
| 6 | 錯誤 | 我明天要考試，下午適合去K書嗎？ | 未知 | 未知 | 1 |
| 7 | 邊界 | 我要報名迎新晚會 (未提供姓名) | 報名 | 迎新晚會 | 1 |
| 8 | 正常 | 報名迎新晚會，我是張三 | 報名 | 迎新晚會 | 1 |

**7-2 流程測試（8～10 步，依序執行）**

| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 王小明 | 我要報名迎新晚會，我叫王小明 | 報名成功，剩餘 5 人 |
| 2 | 王小明 | 我要報名程式工作坊，我叫王小明 | 報名成功，剩餘 3 人 |
| 3 | 王小明 | 我要報名籃球友誼賽，我叫王小明 | 拒絕（已達每人最多報名 2 個活動上限） |
| 4 | 王小明 | 查詢迎新晚會剩餘名額 | 顯示總名額 6 人，已報名 1 人，剩餘 5 人 |
| 5 | 陳大同 | 我和三個朋友報名迎新晚會，我是陳大同 | 拒絕（人數超過 1~3 限制） |
| 6 | 陳大同 | 我和兩個朋友報名迎新晚會，我是陳大同 | 報名成功，剩餘 2 人 |
| 7 | 王小明 | 我要取消迎新晚會，名字是王小明 | 取消成功，釋出名額 |
| 8 | 幹部 | 各活動報名狀況 | 列表顯示各活動統計數據（含 JOIN 與 GROUP BY） |

（若有改用其他模型或有測試未通過，請在此說明原因）

In [49]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#   寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#   也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY_CA') if IN_COLAB else os.environ.get('GROQ_API_KEY_CA')
    client = Groq(api_key=GROQ_API_KEY)
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY_CA 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
REASONING_EFFORT = "low"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [50]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
import sqlite3

def init_db():
    conn = sqlite3.connect('student_activity.db')
    cursor = conn.cursor()

    # 每次執行先刪除舊表，確保環境乾淨
    cursor.execute('DROP TABLE IF EXISTS registrations')
    cursor.execute('DROP TABLE IF EXISTS activities')

    # 建立活動資料表
    cursor.execute('''
        CREATE TABLE activities (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            date TEXT,
            quota INTEGER
        )
    ''')

    # 建立報名紀錄表（包含狀態與人數支援進階 A1, A2）
    cursor.execute('''
        CREATE TABLE registrations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            activity_name TEXT,
            student_name TEXT,
            count INTEGER,
            status TEXT
        )
    ''')

    # 放入初始資料
    initial_acts = [
        ('迎新晚會', '2026-11-20', 6),
        ('程式工作坊', '2026-11-25', 4),
        ('籃球友誼賽', '2026-11-28', 3)
    ]
    cursor.executemany('INSERT INTO activities (name, date, quota) VALUES (?, ?, ?)', initial_acts)

    conn.commit()
    conn.close()
    print("✅ 資料庫與初始資料建立完成（支援進階 A1、A2、A3）。")

init_db()

✅ 資料庫與初始資料建立完成（支援進階 A1、A2、A3）。


In [51]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
def show_tables():
    conn = sqlite3.connect('student_activity.db')
    print("【activities 活動資料表】")
    df_act = pd.read_sql_query('SELECT * FROM activities', conn)
    display(df_act)

    print("\n【registrations 報名紀錄表】")
    df_reg = pd.read_sql_query('SELECT * FROM registrations', conn)
    display(df_reg)
    conn.close()

show_tables()

【activities 活動資料表】


,id,name,date,quota
0,1,迎新晚會,2026-11-20,6
1,2,程式工作坊,2026-11-25,4
2,3,籃球友誼賽,2026-11-28,3



【registrations 報名紀錄表】


,id,activity_name,student_name,count,status


In [52]:
# ============================================================
# 第 4 格：AI 解讀函式（修正版）
# ============================================================
ai_cache = {}

def parse_user_intent(user_input):
    if user_input in ai_cache:
        return ai_cache[user_input]

    prompt = f"""
你是一個系學會活動報名系統的語意解析助理。請分析使用者的一句話，並嚴格提取三個資訊：動作、活動、人數。
1. 動作只能是這五個之一：報名、取消、查詢、統計、未知
   - 如果是查詢特定單一活動的名額（例如「查詢籃球友誼賽剩餘名額」、「查詢迎新晚會剩餘名額」），動作請填「查詢」。
   - 如果是查詢全部活動的整體狀況或各活動狀況（例如「各活動報名狀況」），動作請填「統計」。
2. 活動只能是這四個之一：迎新晚會、程式工作坊、籃球友誼賽、未知（如果說其他活動一律填「未知」）
3. 人數：請從句子中提取包含的總人數（例如「我和兩個朋友」代表 3 人，「我自己」或沒提到數字則填 1）。絕對不可以輸出分數（如 1/8）。
4. 輸出格式必須嚴格遵守：動作,活動,人數（中間用逗號隔開，不要有其他字眼）

使用者說的話是：「{user_input}」
請直接輸出格式：
"""
    reply = "未知,未知,1"
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            reply = response.choices[0].message.content.strip()
            time.sleep(2)
            break
        except Exception as e:
            time.sleep(30)

    # 解析回覆
    parts = reply.split(',')
    action = parts[0].strip() if len(parts) > 0 else "未知"
    activity = parts[1].strip() if len(parts) > 1 else "未知"

    # 用正規表示式萃取人數數字，確保安全
    count_str = parts[2].strip() if len(parts) > 2 else "1"
    match_num = re.search(r'\d+', count_str)
    count = int(match_num.group()) if match_num else 1

    # 嚴格驗證白名單
    if action not in ['報名', '取消', '查詢', '統計']:
        action = '未知'
    if activity not in ['迎新晚會', '程式工作坊', '籃球友誼賽']:
        activity = '未知'

    result = (action, activity, count, reply)
    ai_cache[user_input] = result
    return result

# 測試一次
print("測試解析：", parse_user_intent("查詢籃球友誼賽剩餘名額"))

測試解析： ('查詢', '籃球友誼賽', 1, '查詢,籃球友誼賽,1')


In [53]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
test_cases = [
    ("我要報名迎新晚會，我叫王小明", "報名", "迎新晚會", 1),
    ("我和兩個朋友一起報名程式工作坊，我是陳大同", "報名", "程式工作坊", 3),
    ("我要取消迎新晚會，名字是王小明", "取消", "迎新晚會", 1),
    ("查詢籃球友誼賽剩餘名額", "查詢", "籃球友誼賽", 1),
    ("各活動報名狀況", "統計", "未知", 1),
    ("我明天要考試，下午適合去K書嗎？", "未知", "未知", 1),
    ("我要報名迎新晚會", "報名", "迎新晚會", 1),
    ("報名迎新晚會，我是張三", "報名", "迎新晚會", 1)
]

pass_count = 0
for i, (text, exp_act, exp_act_name, exp_count) in enumerate(test_cases, 1):
    act, act_name, count, raw = parse_user_intent(text)
    is_pass = (act == exp_act)
    if is_pass:
        pass_count += 1
    print(f"測試 #{i}: {'✅ 通過' if is_pass else '❌ 失敗'}")
    print(f"  輸入: {text}")
    print(f"  預期: 動作={exp_act}, 活動={exp_act_name}, 人數={exp_count}")
    print(f"  實際: 動作={act}, 活動={act_name}, 人數={count} (原始: {raw})\n")

print(f"總結：通過解讀測試 {pass_count} / {len(test_cases)} 句")

測試 #1: ✅ 通過
  輸入: 我要報名迎新晚會，我叫王小明
  預期: 動作=報名, 活動=迎新晚會, 人數=1
  實際: 動作=報名, 活動=迎新晚會, 人數=1 (原始: 報名,迎新晚會,1)

測試 #2: ✅ 通過
  輸入: 我和兩個朋友一起報名程式工作坊，我是陳大同
  預期: 動作=報名, 活動=程式工作坊, 人數=3
  實際: 動作=報名, 活動=程式工作坊, 人數=3 (原始: 報名,程式工作坊,3)

測試 #3: ✅ 通過
  輸入: 我要取消迎新晚會，名字是王小明
  預期: 動作=取消, 活動=迎新晚會, 人數=1
  實際: 動作=取消, 活動=迎新晚會, 人數=1 (原始: 取消,迎新晚會,1)

測試 #4: ✅ 通過
  輸入: 查詢籃球友誼賽剩餘名額
  預期: 動作=查詢, 活動=籃球友誼賽, 人數=1
  實際: 動作=查詢, 活動=籃球友誼賽, 人數=1 (原始: 查詢,籃球友誼賽,1)

測試 #5: ✅ 通過
  輸入: 各活動報名狀況
  預期: 動作=統計, 活動=未知, 人數=1
  實際: 動作=統計, 活動=未知, 人數=1 (原始: 統計,未知,1)

測試 #6: ✅ 通過
  輸入: 我明天要考試，下午適合去K書嗎？
  預期: 動作=未知, 活動=未知, 人數=1
  實際: 動作=未知, 活動=未知, 人數=1 (原始: 未知,未知,1)

測試 #7: ✅ 通過
  輸入: 我要報名迎新晚會
  預期: 動作=報名, 活動=迎新晚會, 人數=1
  實際: 動作=報名, 活動=迎新晚會, 人數=1 (原始: 報名,迎新晚會,1)

測試 #8: ✅ 通過
  輸入: 報名迎新晚會，我是張三
  預期: 動作=報名, 活動=迎新晚會, 人數=1
  實際: 動作=報名, 活動=迎新晚會, 人數=1 (原始: 報名,迎新晚會,1)

總結：通過解讀測試 8 / 8 句


In [54]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
def handle_system(action, activity, count, name, user_input=""):
    if not name or name == '未知':
        m = re.search(r'(?:我叫|我是|名字是)\s*([^\s，,]+)', user_input)
        if m:
            name = m.group(1)

    if action == '統計':
        conn = sqlite3.connect('student_activity.db')
        cursor = conn.cursor()
        try:
            cursor.execute('''
                SELECT
                    a.name,
                    a.date,
                    a.quota,
                    COALESCE(SUM(CASE WHEN r.status = '已報名' THEN r.count ELSE 0 END), 0) as booked
                FROM activities a
                LEFT JOIN registrations r ON a.name = r.activity_name
                GROUP BY a.name, a.date, a.quota
            ''')
            rows = cursor.fetchall()
            output = "📊 【各活動報名統計】\n"
            for r in rows:
                act_n, act_date, quota, booked = r[0], r[1], r[2], r[3]
                remaining = quota - booked
                output += f"- {act_n} ({act_date})｜總名額: {quota}｜已報名: {booked}｜剩餘: {remaining}\n"
            return output
        finally:
            conn.close()

    if action == '未知' or activity == '未知':
        return "❌ 系統無法識別您的需求。本系統支援迎新晚會、程式工作坊、籃球友誼賽之報名、取消與查詢。"

    conn = sqlite3.connect('student_activity.db')
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT quota FROM activities WHERE name = ?", (activity,))
        row = cursor.fetchone()
        if not row:
            return f"❌ 找不到活動：{activity}"
        quota = row[0]

        cursor.execute("SELECT SUM(count) FROM registrations WHERE activity_name = ? AND status = '已報名'", (activity,))
        booked_row = cursor.fetchone()
        booked_count = booked_row[0] if booked_row and booked_row[0] is not None else 0

        if action == '查詢':
            remaining = quota - booked_count
            return f"📊 {activity} 總名額 {quota} 人，已報名 {booked_count} 人，剩餘名額 {remaining} 人。"

        if not name or name == '未知':
            return "❌ 操作失敗：請提供您的正確姓名以便登記或核對（例如：我叫王小明）。"

        if action == '報名':
            if count < 1 or count > 3:
                return f"❌ 報名失敗：攜伴人數（{count} 人）不符合規定（每筆限 1～3 人）。"

            if booked_count + count > quota:
                return f"❌ 報名失敗：{activity} 名額不足（剩餘名額 {quota - booked_count} 人）。"

            cursor.execute("SELECT id, status FROM registrations WHERE activity_name = ? AND student_name = ?", (activity, name))
            reg_row = cursor.fetchone()
            if reg_row and reg_row[1] == '已報名':
                return f"⚠️ {name} 您已經報名過 {activity} 了，請勿重複報名。"

            cursor.execute("SELECT COUNT(*) FROM registrations WHERE student_name = ? AND status = '已報名'", (name,))
            user_active_count = cursor.fetchone()[0]
            if user_active_count >= 2:
                return f"❌ 報名失敗：{name} 您目前已報名 2 個活動，已達個人報名上限！"

            cursor.execute("INSERT INTO registrations (activity_name, student_name, count, status) VALUES (?, ?, ?, '已報名')", (activity, name, count))
            conn.commit()
            remaining = quota - (booked_count + count)
            return f"✅ 報名成功！已為 {name}（共 {count} 人）登記 {activity}。該活動剩餘名額：{remaining} 人。"

        elif action == '取消':
            cursor.execute("SELECT id, status FROM registrations WHERE activity_name = ? AND student_name = ? AND status = '已報名'", (activity, name))
            reg_row = cursor.fetchone()
            if not reg_row:
                return f"❌ 取消失敗：找不到 {name} 在 {activity} 的有效報名紀錄。"

            cursor.execute("UPDATE registrations SET status = '已取消' WHERE id = ?", (reg_row[0],))
            conn.commit()
            return f"✅ 身分核對成功！已為 {name} 取消 {activity} 的報名。"

    finally:
        conn.close()

In [55]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
init_db()

print("--- 執行完整流程測試 (8 步) ---")
workflow_tests = [
    ("王小明", "我要報名迎新晚會，我叫王小明"),
    ("王小明", "我要報名程式工作坊，我叫王小明"),
    ("王小明", "我要報名籃球友誼賽，我叫王小明"),
    ("王小明", "查詢迎新晚會剩餘名額"),
    ("陳大同", "我和三個朋友報名迎新晚會，我是陳大同"),
    ("陳大同", "我和兩個朋友報名迎新晚會，我是陳大同"),
    ("王小明", "我要取消迎新晚會，名字是王小明"),
    ("幹部", "各活動報名狀況")
]

for i, (name_hint, text) in enumerate(workflow_tests, 1):
    print(f"\n步驟 #{i} | 說話者: {name_hint} | 內容: {text}")
    act, activity, count, raw = parse_user_intent(text)
    result = handle_system(act, activity, count, name_hint, text)
    print(f"系統回應：\n{result}")

print("\n--- 最終資料庫狀態 ---")
show_tables()

✅ 資料庫與初始資料建立完成（支援進階 A1、A2、A3）。
--- 執行完整流程測試 (8 步) ---

步驟 #1 | 說話者: 王小明 | 內容: 我要報名迎新晚會，我叫王小明
系統回應：
✅ 報名成功！已為 王小明（共 1 人）登記 迎新晚會。該活動剩餘名額：5 人。

步驟 #2 | 說話者: 王小明 | 內容: 我要報名程式工作坊，我叫王小明
系統回應：
✅ 報名成功！已為 王小明（共 1 人）登記 程式工作坊。該活動剩餘名額：3 人。

步驟 #3 | 說話者: 王小明 | 內容: 我要報名籃球友誼賽，我叫王小明
Created dataset file at: .gradio/flagged/dataset1.csv
系統回應：
❌ 報名失敗：王小明 您目前已報名 2 個活動，已達個人報名上限！

步驟 #4 | 說話者: 王小明 | 內容: 查詢迎新晚會剩餘名額
系統回應：
📊 迎新晚會 總名額 6 人，已報名 1 人，剩餘名額 5 人。

步驟 #5 | 說話者: 陳大同 | 內容: 我和三個朋友報名迎新晚會，我是陳大同
系統回應：
❌ 報名失敗：攜伴人數（4 人）不符合規定（每筆限 1～3 人）。

步驟 #6 | 說話者: 陳大同 | 內容: 我和兩個朋友報名迎新晚會，我是陳大同
系統回應：
✅ 報名成功！已為 陳大同（共 3 人）登記 迎新晚會。該活動剩餘名額：2 人。

步驟 #7 | 說話者: 王小明 | 內容: 我要取消迎新晚會，名字是王小明
系統回應：
✅ 身分核對成功！已為 王小明 取消 迎新晚會 的報名。

步驟 #8 | 說話者: 幹部 | 內容: 各活動報名狀況
系統回應：
📊 【各活動報名統計】
- 程式工作坊 (2026-11-25)｜總名額: 4｜已報名: 1｜剩餘: 3
- 籃球友誼賽 (2026-11-28)｜總名額: 3｜已報名: 0｜剩餘: 3
- 迎新晚會 (2026-11-20)｜總名額: 6｜已報名: 3｜剩餘: 3


--- 最終資料庫狀態 ---
【activities 活動資料表】


,id,name,date,quota
0,1,迎新晚會,2026-11-20,6
1,2,程式工作坊,2026-11-25,4
2,3,籃球友誼賽,2026-11-28,3



【registrations 報名紀錄表】


,id,activity_name,student_name,count,status
0,1,迎新晚會,王小明,1,已取消
1,2,程式工作坊,王小明,1,已報名
2,3,迎新晚會,陳大同,3,已報名


In [48]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
def chatbot_interface(user_input):
    act, activity, count, raw = parse_user_intent(user_input)
    m = re.search(r'(?:我叫|我是|名字是)\s*([^\s，,]+)', user_input)
    name = m.group(1) if m else "未知"

    response = handle_system(act, activity, count, name, user_input)
    return f"【AI 解析】動作: {act} | 活動: {activity} | 人數: {count}\n【系統回應】\n{response}"

demo = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(label="請輸入您的需求（例如：我要報名迎新晚會，我叫王小明）"),
    outputs=gr.Textbox(label="系統處理結果"),
    title="🎉 系學會活動報名與查詢系統",
    description="提供迎新晚會、程式工作坊、籃球友誼賽之報名、取消、查詢及統計服務！"
)

if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://915452849623453fcf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
